In [ ]:
%%writefile byte_aligned_test.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <time.h>

static const char decode_table[256] = {
    ['A']='A', ['a']='A', ['C']='C', ['c']='C',
    ['G']='G', ['g']='G', ['T']='T', ['t']='T',
    ['N']='N', ['n']='N',
};
static inline char decode_char(uint8_t v) {
    char c = decode_table[v];
    return c ? c : '?';
}

static inline void pad_sequences(uint8_t *ref, uint8_t *read, int len) {
    int rem = len & 63;
    if (rem) {
        int pad = 64 - rem;
        memset(read + len, 0xFF, pad);
        memset(ref  + len, 0x00, pad);
    }
}

void print_sequence_debug(const char *label,
                           const uint8_t *ref, const uint8_t *read,
                           int len, int kmer_size, int threshold, int show_len)
{
    if (show_len > len) show_len = len;
    printf("\nSEQUENCE DEBUG: %s\n", label);
    printf("Length=%d  KmerSize=%d  Threshold=%d  Showing first %d bases\n",
           len, kmer_size, threshold, show_len);
    int total_mm = 0;
    for (int start = 0; start < show_len; start += 64) {
        int end = start + 64;
        if (end > show_len) end = show_len;
        printf("\n  Pos  [%4d - %4d]\n", start, end - 1);
        printf("  Kmer  ");
        for (int i = start; i < end; i++) printf("%c", (i % kmer_size == 0) ? '|' : '-');
        printf("\n  REF   ");
        for (int i = start; i < end; i++) printf("%c", decode_char(ref[i]));
        printf("\n  READ  ");
        for (int i = start; i < end; i++) printf("%c", decode_char(read[i]));
        printf("\n  MATCH ");
        for (int i = start; i < end; i++) printf("%c", ref[i] == read[i] ? '1' : '0');
        printf("\n  DIFF  ");
        int mm = 0;
        for (int i = start; i < end; i++) {
            if (ref[i] != read[i]) { printf("^"); mm++; } else printf(" ");
        }
        printf("  (%d mismatches)\n", mm);
        total_mm += mm;
    }
    printf("\n  SUMMARY: bases=%d  mismatches=%d  match=%.1f%%\n\n",
           show_len, total_mm, 100.0 * (show_len - total_mm) / show_len);
}

int SneakySnake_C(int ReadLength, uint8_t* RefSeq, uint8_t* ReadSeq,
                  int EditThreshold, int KmerSize, int IterationNo)
{
    int n, e, K, index = 0, Edits = 0, count = 0, GlobalCount = 0;
    int KmerStart = 0, KmerEnd = 0, roundsNo = 0;
    for (K = 0; K < (ReadLength / KmerSize); K++) {
        KmerStart = K * KmerSize;
        KmerEnd   = (K < (ReadLength / KmerSize) - 1) ? (K+1)*KmerSize : ReadLength;
        index = KmerStart; roundsNo = 1;
        while (index < KmerEnd) {
            GlobalCount = 0;
            for (n = index; n < KmerEnd; n++) {
                if (ReadSeq[n] != RefSeq[n]) goto EXIT1; else GlobalCount++;
            } EXIT1:
            if (GlobalCount == (KmerEnd - KmerStart)) goto LOOP;
            for (e = 1; e <= EditThreshold; e++) {
                count = 0;
                for (n = index; n < KmerEnd; n++) {
                    if (n < e) goto EXIT2;
                    else if (ReadSeq[n-e] != RefSeq[n]) goto EXIT2; else count++;
                } EXIT2:
                if (count > GlobalCount) GlobalCount = count;
                if (count == (KmerEnd - KmerStart)) goto LOOP;
                count = 0;
                for (n = index; n < KmerEnd; n++) {
                    if (n > ReadLength - e - 1) goto EXIT3;
                    else if (ReadSeq[n+e] != RefSeq[n]) goto EXIT3; else count++;
                } EXIT3:
                if (count > GlobalCount) GlobalCount = count;
                if (count == (KmerEnd - KmerStart)) goto LOOP;
            }
            index += GlobalCount;
            if (index < KmerEnd) { Edits++; index++; }
            if (roundsNo > IterationNo) goto LOOP;
            roundsNo++;
            if (Edits > EditThreshold) return 0;
        }
        LOOP:
        if (Edits > EditThreshold) return 0;
    }
    return 1;
}

extern uint64_t SneakySnake(uint64_t ReadLength, uint8_t* RefSeq,
                             uint8_t* ReadSeq, uint64_t EditThreshold,
                             uint64_t IterationNo, uint64_t KmerSize);
extern uint64_t current_position, current_edits, mismatch_count, safety_counter;

int calculate_edit_distance(const char* s1, const char* s2, int len) {
    if (len > 2000) return 0;
    int dp[len+1][len+1];
    for (int i = 0; i <= len; i++) dp[i][0] = i;
    for (int j = 0; j <= len; j++) dp[0][j] = j;
    for (int i = 1; i <= len; i++)
        for (int j = 1; j <= len; j++) {
            if (s1[i-1] == s2[j-1]) dp[i][j] = dp[i-1][j-1];
            else { int a=dp[i-1][j],b=dp[i][j-1],c=dp[i-1][j-1],m=a<b?a:b; dp[i][j]=1+(m<c?m:c); }
        }
    return dp[len][len];
}

int main(int argc, char *argv[]) {
    if (argc < 5) {
        printf("Usage: %s <file> <threshold> <seq_len> <kmer_size>"
               " [limit] [debug_limit] [debug_show_len]\n", argv[0]);
        return 1;
    }
    const char* filename = argv[1];
    int threshold    = atoi(argv[2]);
    int fixed_len    = atoi(argv[3]);
    int kmer_size    = atoi(argv[4]);
    int limit        = (argc >= 6) ? atoi(argv[5]) : 30000;
    int debug_limit  = (argc >= 7) ? atoi(argv[6]) : 5;
    int debug_show   = (argc >= 8) ? atoi(argv[7]) : 128;

    if (kmer_size <= 0 || kmer_size > fixed_len) {
        printf("Error: Invalid kmer_size %d\n", kmer_size); return 1;
    }

    FILE *file = fopen(filename, "r");
    if (!file) { printf("Error: Cannot open %s\n", filename); return 1; }
    setvbuf(file, NULL, _IOFBF, 8*1024*1024);

    printf("Loading (FixedLen=%d, KmerSize=%d, Limit=%d)...\n",
           fixed_len, kmer_size, limit);
    size_t stride = ((size_t)(fixed_len + 64) + 63) & ~63ULL;

    char    **raw_read  = malloc(limit * sizeof(char*));
    char    **raw_ref   = malloc(limit * sizeof(char*));
    uint8_t **read_enc  = malloc(limit * sizeof(uint8_t*));
    uint8_t **ref_enc   = malloc(limit * sizeof(uint8_t*));
    int      *lengths   = malloc(limit * sizeof(int));
    int      *edit_dists= malloc(limit * sizeof(int));
    if (!raw_read || !raw_ref || !read_enc || !ref_enc || !lengths || !edit_dists) {
        perror("malloc"); return 1;
    }

    char *line = NULL; size_t cap = 0;
    int count = 0, skipped = 0;

    while (getline(&line, &cap, file) != -1 && count < limit) {
        char *read_seq = line;
        char *ref_seq  = strpbrk(line, "\t ");
        if (!ref_seq) continue;
        *ref_seq++ = '\0';
        while (*ref_seq == ' ' || *ref_seq == '\t') ref_seq++;
        ref_seq[strcspn(ref_seq, "\r\n")] = 0;
        if ((int)strlen(read_seq) < fixed_len) { skipped++; continue; }

        int len = fixed_len;
        lengths[count]    = len;
        raw_read[count]   = strndup(read_seq, len);
        raw_ref[count]    = strndup(ref_seq,  len);
        edit_dists[count] = (count < debug_limit)
                            ? calculate_edit_distance(read_seq, ref_seq, len) : -1;

        uint8_t *rdbuf = aligned_alloc(64, stride);
        uint8_t *rfbuf = aligned_alloc(64, stride);
        if (!rdbuf || !rfbuf) { perror("aligned_alloc"); return 1; }
        memset(rdbuf, 0xFF, stride);
        memset(rfbuf, 0x00, stride);
        for (int j = 0; j < len; j++) {
            rdbuf[j] = (uint8_t)(read_seq[j] & 0xDF);
            rfbuf[j] = (uint8_t)(ref_seq[j]  & 0xDF);
        }
        pad_sequences(rfbuf, rdbuf, len);
        read_enc[count] = rdbuf;
        ref_enc[count]  = rfbuf;

        count++;
        if (count % 500 == 0) { printf("Loaded %d...\r", count); fflush(stdout); }
    }
    fclose(file); free(line);
    if (skipped) printf("\nSkipped %d sequences shorter than %d bp.\n", skipped, fixed_len);
    printf("\nLoaded %d sequences.\n", count);

    printf("\n=== DEBUG: FIRST %d SEQUENCES ===\n", debug_limit);
    for (int i = 0; i < debug_limit && i < count; i++) {
        printf("\n--- Pair #%d  TrueEditDist=%d  Threshold=%d ---\n",
               i, edit_dists[i], threshold);
        int show = debug_show < lengths[i] ? debug_show : lengths[i];
        printf("  REF : %.*s\n  READ: %.*s\n", show, raw_ref[i], show, raw_read[i]);

        char lbl[32]; snprintf(lbl, sizeof(lbl), "Pair #%d", i);
        print_sequence_debug(lbl, ref_enc[i], read_enc[i],
                             lengths[i], kmer_size, threshold, debug_show);

        int c_res = SneakySnake_C(lengths[i], ref_enc[i], read_enc[i],
                                  threshold, kmer_size, lengths[i]);
        current_position = current_edits = mismatch_count = safety_counter = 0;
        int a_res = (int)SneakySnake((uint64_t)lengths[i], ref_enc[i], read_enc[i],
                                      (uint64_t)threshold, (uint64_t)(lengths[i]),
                                      (uint64_t)kmer_size);
        printf("  C  : %s\n  ASM: %s\n",
               c_res ? "ACCEPT" : "REJECT", a_res ? "ACCEPT" : "REJECT");
        if (c_res != a_res)
            printf("  *** DISAGREE C=%d ASM=%d ***\n", c_res, a_res);
        else
            printf("  Agreement: OK\n");
    }

    printf("\n=== BENCHMARK: reads=%d  len=%d  threshold=%d  kmer=%d ===\n",
           count, fixed_len, threshold, kmer_size);

    int c_matches = 0, asm_matches = 0;

    clock_t t0 = clock();
    for (int i = 0; i < count; i++)
        if (SneakySnake_C(lengths[i], ref_enc[i], read_enc[i],
                          threshold, kmer_size, lengths[i]))
            c_matches++;
    double c_time = (double)(clock()-t0) / CLOCKS_PER_SEC;

    clock_t t1 = clock();
    for (int i = 0; i < count; i++) {
        current_position = current_edits = mismatch_count = safety_counter = 0;
        if ((int)SneakySnake((uint64_t)lengths[i], ref_enc[i], read_enc[i],
                              (uint64_t)threshold, (uint64_t)(lengths[i]),
                              (uint64_t)kmer_size))
            asm_matches++;
    }
    double asm_time = (double)(clock()-t1) / CLOCKS_PER_SEC;

    printf("\nC   : accepted=%d  time=%.4fs\n", c_matches,   c_time);
    printf("ASM : accepted=%d  time=%.4fs\n",   asm_matches, asm_time);
    if (asm_time > 0) printf("Speedup: %.2fx\n", c_time / asm_time);

    if (c_matches != asm_matches)
        printf("\nWARNING: DISAGREE C=%d ASM=%d  diff=%d\n",
               c_matches, asm_matches, abs(c_matches-asm_matches));
    else
        printf("\nOK: C and ASM agree — %d reads accepted.\n", c_matches);

    for (int i = 0; i < count; i++) {
        free(raw_read[i]); free(raw_ref[i]);
        free(read_enc[i]); free(ref_enc[i]);
    }
    free(raw_read); free(raw_ref);
    free(read_enc); free(ref_enc);
    free(lengths);  free(edit_dists);
    return 0;
}

In [ ]:
%%writefile byte_aligned_test.asm

default rel
bits 64

%define VCMP_NEQ 4

section .data

global SneakySnake
global current_position
global current_edits
global mismatch_count
global safety_counter

align 64
current_position dq 0
current_edits    dq 0
mismatch_count   dq 0
safety_counter   dq 0

section .text
global SneakySnake

; SneakySnake(ReadLength, RefSeq, ReadSeq, EditThreshold,
;             IterationNo, KmerSize)
;
; RDI = ReadLength      RSI = RefSeq     RDX = ReadSeq
; RCX = EditThreshold   R8  = IterationNo R9  = KmerSize
;
; Registers:
;   r13 = ReadLength
;   r12 = RefSeq
;   r11 = ReadSeq
;   r10 = EditThreshold
;   rbx = KmerEnd  (advances by KmerSize each outer iteration)
;
; Stack (qwords at [rbp-N]):
;    8 = KmerSize
;   16 = Edits
;   24 = roundsNo
;   32 = IterationNo
SneakySnake:
    push    rbp
    mov     rbp, rsp
    push    rbx
    push    r12
    push    r13
    push    r14
    push    r15
    sub     rsp, 32

    mov     r13, rdi
    mov     r12, rsi
    mov     r11, rdx
    mov     r10, rcx
    mov     [rbp-32], r8       
    mov     [rbp-8],  r9      

    mov     qword [rbp-16], 0  

    mov     rbx, r9
    cmp     rbx, r13
    cmova   rbx, r13

    mov     r15, 0             
    mov     qword [rbp-24], 1  

.while_loop:
    cmp     r15, rbx
    jae     .kmer_next

    mov     rax, [rbp-16]
    cmp     rax, r10
    jg      .reject

    xor     r14, r14
    mov     rax, r15
    mov     rcx, rbx
    sub     rcx, rax          

    prefetcht0 [r11 + rax + 512]
    prefetcht0 [r12 + rax + 512]

.main_256:
    cmp     rcx, 256
    jb      .main_64

    prefetcht0 [r11 + rax + 768]
    prefetcht0 [r12 + rax + 768]

    vmovdqu8 zmm0, [r11 + rax]
    vmovdqu8 zmm1, [r12 + rax]
    vpcmpub  k1, zmm0, zmm1, VCMP_NEQ
    ktestq   k1, k1
    jnz      .main_hit_0

    vmovdqu8 zmm2, [r11 + rax + 64]
    vmovdqu8 zmm3, [r12 + rax + 64]
    vpcmpub  k2, zmm2, zmm3, VCMP_NEQ
    ktestq   k2, k2
    jnz      .main_hit_64

    vmovdqu8 zmm4, [r11 + rax + 128]
    vmovdqu8 zmm5, [r12 + rax + 128]
    vpcmpub  k3, zmm4, zmm5, VCMP_NEQ
    ktestq   k3, k3
    jnz      .main_hit_128

    vmovdqu8 zmm6, [r11 + rax + 192]
    vmovdqu8 zmm7, [r12 + rax + 192]
    vpcmpub  k4, zmm6, zmm7, VCMP_NEQ
    ktestq   k4, k4
    jnz      .main_hit_192

    add     r14, 256
    add     rax, 256
    sub     rcx, 256
    jmp     .main_256

.main_hit_0:
    kmovq   rdx, k1
    tzcnt   rdx, rdx
    add     r14, rdx
    jmp     .main_done
.main_hit_64:
    kmovq   rdx, k2
    tzcnt   rdx, rdx
    add     r14, 64
    add     r14, rdx
    jmp     .main_done
.main_hit_128:
    kmovq   rdx, k3
    tzcnt   rdx, rdx
    add     r14, 128
    add     r14, rdx
    jmp     .main_done
.main_hit_192:
    kmovq   rdx, k4
    tzcnt   rdx, rdx
    add     r14, 192
    add     r14, rdx
    jmp     .main_done

.main_64:
    cmp     rcx, 64
    jb      .main_tail

    vmovdqu8 zmm0, [r11 + rax]
    vmovdqu8 zmm1, [r12 + rax]
    vpcmpub  k1, zmm0, zmm1, VCMP_NEQ
    ktestq   k1, k1
    jnz      .main_hit_single

    add     r14, 64
    add     rax, 64
    sub     rcx, 64
    jmp     .main_64

.main_hit_single:
    kmovq   rdx, k1
    tzcnt   rdx, rdx
    add     r14, rdx
    jmp     .main_done

.main_tail:
    test    rcx, rcx
    jz      .main_done

    mov     rdi, rcx           
    mov     rdx, -1
    bzhi    rdx, rdx, rcx
    kmovq   k1, rdx

    vmovdqu8 zmm0 {k1}{z}, [r11 + rax]
    vmovdqu8 zmm1 {k1}{z}, [r12 + rax]
    vpcmpub  k2 {k1}, zmm0, zmm1, VCMP_NEQ
    ktestq   k2, k2
    jz      .main_tail_all
    kmovq   rcx, k2
    tzcnt   rcx, rcx
    add     r14, rcx
    jmp     .main_done
.main_tail_all:
    add     r14, rdi

.main_done:
    mov     rcx, rbx           
    sub     rcx, r15
    cmp     r14, rcx
    jae     .kmer_next

    mov     r8, 1

.diag_loop:
    cmp     r8, r10
    ja      .diag_exhausted

    mov     r9,  r15
    mov     rsi, r15
    sub     rsi, r8           

    cmp     r15, r8
    jb      .upper_skip      

    xor     rax, rax

    mov     rdx, rbx
    sub     rdx, r9
    mov     rdi, r13
    sub     rdi, rsi
    cmp     rdx, rdi
    cmovg   rdx, rdi

.upper_vector:
    cmp     rdx, 64
    jb      .upper_tail

    prefetcht0 [r11 + rsi + 256]
    prefetcht0 [r12 + r9  + 256]

    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    vpcmpub  k1, zmm0, zmm1, VCMP_NEQ
    ktestq   k1, k1
    jnz      .upper_hit

    add     rax, 64
    add     r9,  64
    add     rsi, 64
    sub     rdx, 64
    jmp     .upper_vector

.upper_hit:
    kmovq   rcx, k1
    tzcnt   rcx, rcx
    add     rax, rcx
    jmp     .upper_cmp

.upper_tail:
    test    rdx, rdx
    jz      .upper_cmp

    mov     rcx, -1
    bzhi    rcx, rcx, rdx
    kmovq   k1, rcx

    vmovdqu8 zmm0 {k1}{z}, [r11 + rsi]
    vmovdqu8 zmm1 {k1}{z}, [r12 + r9]
    vpcmpub  k2 {k1}, zmm0, zmm1, VCMP_NEQ
    ktestq   k2, k2
    jz      .upper_tail_all
    kmovq   rcx, k2
    tzcnt   rcx, rcx
    add     rax, rcx
    jmp     .upper_cmp
.upper_tail_all:
    add     rax, rdx

.upper_cmp:
    cmp     rax, r14
    jbe     .check_lower
    mov     r14, rax
    mov     rcx, rbx
    sub     rcx, r15
    cmp     r14, rcx
    jae     .kmer_next

.upper_skip:

.check_lower:
    mov     r9,  r15
    lea     rsi, [r15 + r8]    

    cmp     rsi, r13
    jae     .lower_skip

    xor     rax, rax

    mov     rdx, rbx
    sub     rdx, r9
    mov     rdi, r13
    sub     rdi, rsi
    cmp     rdx, rdi
    cmovg   rdx, rdi

.lower_vector:
    cmp     rdx, 64
    jb      .lower_tail

    prefetcht0 [r11 + rsi + 256]
    prefetcht0 [r12 + r9  + 256]

    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + r9]
    vpcmpub  k1, zmm0, zmm1, VCMP_NEQ
    ktestq   k1, k1
    jnz      .lower_hit

    add     rax, 64
    add     r9,  64
    add     rsi, 64
    sub     rdx, 64
    jmp     .lower_vector

.lower_hit:
    kmovq   rcx, k1
    tzcnt   rcx, rcx
    add     rax, rcx
    jmp     .lower_cmp

.lower_tail:
    test    rdx, rdx
    jz      .lower_cmp

    mov     rcx, -1
    bzhi    rcx, rcx, rdx
    kmovq   k1, rcx

    vmovdqu8 zmm0 {k1}{z}, [r11 + rsi]
    vmovdqu8 zmm1 {k1}{z}, [r12 + r9]
    vpcmpub  k2 {k1}, zmm0, zmm1, VCMP_NEQ
    ktestq   k2, k2
    jz      .lower_tail_all
    kmovq   rcx, k2
    tzcnt   rcx, rcx
    add     rax, rcx
    jmp     .lower_cmp
.lower_tail_all:
    add     rax, rdx

.lower_cmp:
    cmp     rax, r14
    jbe     .next_diag
    mov     r14, rax
    mov     rcx, rbx
    sub     rcx, r15
    cmp     r14, rcx
    jae     .kmer_next

.lower_skip:
.next_diag:
    inc     r8
    jmp     .diag_loop

.diag_exhausted:
    add     r15, r14
    cmp     r15, rbx
    jae     .kmer_next

    inc     qword [rbp-16]     
    inc     r15

    mov     rax, [rbp-16]
    cmp     rax, r10
    jg      .reject

    inc     qword [rbp-24]     
    mov     rax, [rbp-24]
    cmp     rax, [rbp-32]      
    ja      .kmer_next

    jmp     .while_loop

.kmer_next:
    mov     rax, [rbp-16]
    cmp     rax, r10
    jg      .reject

    mov     r15, rbx           
    add     rbx, [rbp-8]      
    cmp     rbx, r13
    cmova   rbx, r13

    cmp     r15, r13          
    jae     .accept

    mov     qword [rbp-24], 1  
    jmp     .while_loop

.accept:
    mov     [current_position], r15
    mov     rax, [rbp-16]
    mov     [current_edits], rax
    cmp     rax, r10
    jg      .reject_w
    mov     eax, 1
    jmp     .end

.reject:
    mov     [current_position], r15
    mov     rax, [rbp-16]
    mov     [current_edits], rax
.reject_w:
    xor     eax, eax

.end:
    vzeroupper
    add     rsp, 32
    pop     r15
    pop     r14
    pop     r13
    pop     r12
    pop     rbx
    leave
    ret

In [ ]:
!nasm -f elf64 byte_aligned_test.asm -o byte_aligned_test.o 
!gcc -c byte_aligned_test.c -o byte_aligned_test_c.o -mavx512f -mavx512bw 
!gcc byte_aligned_test.o byte_aligned_test_c.o -o byte_aligned -mavx512f -mavx512bw 

# ERR E2-E40

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 1 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 2 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 3 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 4 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 5 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 6 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 7 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 8 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 9 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 10 100 100 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 1 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 2 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 3 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 4 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 5 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 6 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 7 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 8 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 9 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 10 100 100 30000 0

# SRR E8-E100

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 2 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 5 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 7 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 10 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 12 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 15 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 17 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 20 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 22 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 25 100 100 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 2 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 5 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 7 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 10 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 12 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 15 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 17 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 20 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 22 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/SRR826471_1_E100_30million.txt" 25 100 100 30000 0

## LONG SEQUENCES 10K-100K

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 100 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 200 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 300 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 400 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 500 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 600 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 700 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 800 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 900 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 1000 100 100 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 0 1 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 1000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 2000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 3000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 4000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 5000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 6000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 7000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 8000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 9000 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 10000 100 100 30000 0

## FRUITFLY 100 - 100k

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 1 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 2 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 3 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 4 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 5 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 6 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 7 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 8 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 9 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100bp.txt" 10 100 100 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 0 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 2 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 5 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 7 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 10 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 12 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 15 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 17 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 20 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 22 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_250bp.txt" 25 250 250 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 0 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 10 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 20 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 30 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 40 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 50 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 60 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 70 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 80 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 90 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_1kbp.txt" 100 1000 1000 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 0 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 100 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 200 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 300 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 400 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 500 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 600 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 700 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 800 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 900 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_10kbp.txt" 1000 10000 10000 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 0 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 1000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 2000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 3000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 4000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 5000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 6000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 7000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 8000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 9000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/FruitFly_Reads_100kbp.txt" 10000 100000 100000 30000 0

## ECOLI 100 - 100k

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 0 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 1 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 2 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 3 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 4 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 5 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 6 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 7 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 8 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 9 100 100 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100bp.txt" 10 100 100 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 0 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 2 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 5 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 7 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 10 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 12 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 15 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 17 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 20 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 22 250 250 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_250bp.txt" 25 250 250 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 0 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 10 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 20 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 30 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 40 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 50 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 60 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 70 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 80 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 90 1000 1000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_1kbp.txt" 100 1000 1000 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 0 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 100 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 200 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 300 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 400 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 500 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 600 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 700 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 800 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 900 10000 10000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_10kbp.txt" 1000 10000 10000 30000 0

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 0 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 1000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 2000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 3000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 4000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 5000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 6000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 7000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 8000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 9000 100000 100000 30000 0
!./byte_aligned "../THES3/DNAPAIRS/Ecoli_Reads_100kbp.txt" 10000 100000 100000 30000 0

In [ ]:
SneakySnake(ReadLength R, RefSeq, ReadSeq, EditThreshold E, IterationNo I, KmerSize K):

  index    = 0           // KmerStart
  KmerEnd  = min(K, R)   // rbx
  Edits    = 0
  roundsNo = 1

  while index < KmerEnd:                          // outer kmer loop
    if Edits > E: return REJECT

    GlobalCount = vectorScan(ReadSeq, RefSeq,     // AVX-512 scan
                    index, KmerEnd)               // O(K/64) per call

    if GlobalCount >= (KmerEnd - index):
      goto kmer_next                              // whole kmer matched

    shift = 1
    while shift <= E:                             // diagonal loop
      upper = vectorScan(ReadSeq[index-shift],
                         RefSeq[index],
                         KmerEnd - index)         // O(K/64)
      lower = vectorScan(ReadSeq[index+shift],
                         RefSeq[index],
                         KmerEnd - index)         // O(K/64)

      GlobalCount = max(GlobalCount, upper, lower)
      if GlobalCount >= (KmerEnd - index):
        goto kmer_next
      shift++
    // diag_exhausted:
    index += GlobalCount
    if index < KmerEnd:
      Edits++; index++
      roundsNo++
      if roundsNo > I: goto kmer_next
    // continue while_loop

  kmer_next:
    if Edits > E: return REJECT
    index   = KmerEnd
    KmerEnd = min(KmerEnd + K, R)
    roundsNo = 1
    if index >= R: return ACCEPT
    // repeat while_loop

  return ACCEPT

In [ ]:
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_6666bp.txt"   0  6666 6666 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_6666bp.txt"   6665  6666 6666 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_13332bp.txt"  13331 13332 13332 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_19998bp.txt"  19997 19998 19998 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_26664bp.txt"  26663 26664 26664 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_33330bp.txt"  33329 33330 33330 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_39996bp.txt"  39995 39996 39996 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_46662bp.txt"  46661 46662 46662 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_53328bp.txt"  53327 53328 53328 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_59994bp.txt"  59993 59994 59994 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_66660bp.txt"  66659 66660 66660 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_73326bp.txt"  73325 73326 73326 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_79992bp.txt"  79991 79992 79992 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_86658bp.txt"  86657 86658 86658 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_93324bp.txt"  93323 93324 93324 1000 0
!./byte_aligned "../THES3/DNAPAIRS/WorstCase/WorstCase_100000bp.txt" 99999 100000 100000 1000 0